# Setup Athena Query Engine

In [94]:
import boto3
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket = sess.default_bucket()

In [95]:
from pyathena import connect
import awswrangler as wr

In [96]:
aerodelay_db = "aerodelay"

In [97]:
aerodelay_s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [98]:
%store aerodelay_db aerodelay_s3_staging_dir

Stored 'aerodelay_db' (str)
Stored 'aerodelay_s3_staging_dir' (str)


### SQL Execution Helper Function

In [8]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [99]:
def run_select(sql):
    return wr.athena.read_sql_query(
        sql=sql,
        database=database,
        s3_output=aerodelay_s3_staging_dir
    )

In [100]:
def run_ddl(sql):
    return wr.athena.start_query_execution(
        sql=sql,
        database=database,
        s3_output=aerodelay_s3_staging_dir
    )

### Create DB

In [101]:
statement = "CREATE DATABASE IF NOT EXISTS {}".format(aerodelay_db)
print(statement)

CREATE DATABASE IF NOT EXISTS aerodelay


In [102]:
# create db
run_ddl(statement)

'5c7283be-ed86-47fb-b0e8-326f1c5d9d09'

### Setup Table

In [103]:
# Set Athena parameters
aerodelay_table = "flights"

In [104]:
%store aerodelay_table

Stored 'aerodelay_table' (str)


In [105]:
%store -r s3_aerodelay

In [23]:
s3_aerodelay

's3://sagemaker-us-east-1-103012382341/airline-delay'

In [107]:
stmt = """CREATE EXTERNAL TABLE IF NOT EXISTS {}.{} (
      Year INT, Quarter INT, Month INT, DayofMonth INT, DayOfWeek INT,
      FlightDate STRING, Reporting_Airline STRING,
      DOT_ID_Reporting_Airline INT, IATA_CODE_Reporting_Airline STRING,
      Tail_Number STRING, Flight_Number_Reporting_Airline INT,
      OriginAirportID INT, OriginAirportSeqID INT, OriginCityMarketID INT,
      Origin STRING, OriginCityName STRING, OriginState STRING,
      OriginStateFips INT, OriginStateName STRING, OriginWac INT,
      DestAirportID INT, DestAirportSeqID INT, DestCityMarketID INT,
      Dest STRING, DestCityName STRING, DestState STRING,
      DestStateFips INT, DestStateName STRING, DestWac INT,
      CRSDepTime INT, DepTime DOUBLE, DepDelay DOUBLE,
      DepDelayMinutes DOUBLE, DepDel15 DOUBLE, DepartureDelayGroups INT,
      DepTimeBlk STRING, TaxiOut DOUBLE, WheelsOff DOUBLE,
      WheelsOn DOUBLE, TaxiIn DOUBLE, CRSArrTime INT,
      ArrTime DOUBLE, ArrDelay DOUBLE, ArrDelayMinutes DOUBLE,
      ArrDel15 DOUBLE, ArrivalDelayGroups INT, ArrTimeBlk STRING,
      Cancelled DOUBLE, CancellationCode STRING, Diverted DOUBLE,
      CRSElapsedTime DOUBLE, ActualElapsedTime DOUBLE, AirTime DOUBLE,
      Flights DOUBLE, Distance DOUBLE, DistanceGroup INT,
      CarrierDelay DOUBLE, WeatherDelay DOUBLE, NASDelay DOUBLE,
      SecurityDelay DOUBLE, LateAircraftDelay DOUBLE
  )
  ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
  WITH SERDEPROPERTIES (
      "separatorChar" = ",",
      "quoteChar"     = "\\\""
  )
  STORED AS TEXTFILE
  LOCATION '{}/raw/'
  TBLPROPERTIES ('skip.header.line.count'='1')
""".format(aerodelay_db, aerodelay_table, s3_aerodelay)

In [108]:
stmt

'CREATE EXTERNAL TABLE IF NOT EXISTS aerodelay.flights (\n      Year INT, Quarter INT, Month INT, DayofMonth INT, DayOfWeek INT,\n      FlightDate STRING, Reporting_Airline STRING,\n      DOT_ID_Reporting_Airline INT, IATA_CODE_Reporting_Airline STRING,\n      Tail_Number STRING, Flight_Number_Reporting_Airline INT,\n      OriginAirportID INT, OriginAirportSeqID INT, OriginCityMarketID INT,\n      Origin STRING, OriginCityName STRING, OriginState STRING,\n      OriginStateFips INT, OriginStateName STRING, OriginWac INT,\n      DestAirportID INT, DestAirportSeqID INT, DestCityMarketID INT,\n      Dest STRING, DestCityName STRING, DestState STRING,\n      DestStateFips INT, DestStateName STRING, DestWac INT,\n      CRSDepTime INT, DepTime DOUBLE, DepDelay DOUBLE,\n      DepDelayMinutes DOUBLE, DepDel15 DOUBLE, DepartureDelayGroups INT,\n      DepTimeBlk STRING, TaxiOut DOUBLE, WheelsOff DOUBLE,\n      WheelsOn DOUBLE, TaxiIn DOUBLE, CRSArrTime INT,\n      ArrTime DOUBLE, ArrDelay DOUBLE,

In [109]:
# create table
run_ddl(stmt)

'4a5aa2ce-b3c4-4c99-a888-4557a1d4f624'

In [110]:
import awswrangler as wr

get_data = "SELECT * FROM aerodelay.flights LIMIT 10"
run_select(get_data)

[05/27/26 00:32:25] INFO     Created CTAS table                                                       ]8;id=12244303;file:///opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py\_utils.py]8;;\:]8;id=12244304;file:///opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py#891\891]8;;\
                             "aerodelay"."temp_table_651288a8e289425e8b5d66b2f0aef2d2"                             

,year,quarter,month,dayofmonth,dayofweek,flightdate,reporting_airline,dot_id_reporting_airline,iata_code_reporting_airline,tail_number,...,actualelapsedtime,airtime,flights,distance,distancegroup,carrierdelay,weatherdelay,nasdelay,securitydelay,lateaircraftdelay
0,2024,1,3,7,4,2024-03-07,DL,19790,DL,N351DN,...,96.0,45.0,1.0,236.0,1,18.0,0.0,23.0,0.0,0.0
1,2024,1,3,8,5,2024-03-08,DL,19790,DL,N370DN,...,85.0,45.0,1.0,236.0,1,NaN,NaN,NaN,NaN,NaN
2,2024,1,3,10,7,2024-03-10,DL,19790,DL,N376DN,...,270.0,244.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN
3,2024,1,3,11,1,2024-03-11,DL,19790,DL,N351DN,...,246.0,221.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN
4,2024,1,3,12,2,2024-03-12,DL,19790,DL,N381DZ,...,251.0,221.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN
5,2024,1,3,13,3,2024-03-13,DL,19790,DL,N316DN,...,228.0,211.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN
6,2024,1,3,14,4,2024-03-14,DL,19790,DL,N395DZ,...,278.0,226.0,1.0,1947.0,8,0.0,0.0,25.0,0.0,0.0
7,2024,1,3,15,5,2024-03-15,DL,19790,DL,N121DZ,...,245.0,222.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN
8,2024,1,3,16,6,2024-03-16,DL,19790,DL,N342DN,...,241.0,216.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN
9,2024,1,3,17,7,2024-03-17,DL,19790,DL,N345DN,...,255.0,218.0,1.0,1947.0,8,NaN,NaN,NaN,NaN,NaN


In [111]:
wr.catalog.tables(database="aerodelay")

,Database,Table,Description,TableType,Columns,Partitions
0,aerodelay,flights,,EXTERNAL_TABLE,"year, quarter, month, dayofmonth, dayofweek, f...",


In [112]:
%store

Stored variables and their in-db values:
aerodelay_db                         -> 'aerodelay'
aerodelay_s3_staging_dir             -> 's3://sagemaker-us-east-1-103012382341/athena/stag
aerodelay_table                      -> 'flights'
s3_aerodelay                         -> 's3://sagemaker-us-east-1-103012382341/airline-del
